Downloaded Dataset

In [ ]:
!pip install -q gdown

import gdown

file_id = "1IqSe8lgyIDt_WcTK9BT5gdTmAoIj0J7R"
url = f"https://drive.google.com/uc?id={file_id}"

output = "/content/RDD2022-India.v5-rdd2022-india.yolov8.zip"

gdown.download(url, output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1IqSe8lgyIDt_WcTK9BT5gdTmAoIj0J7R
From (redirected): https://drive.google.com/uc?id=1IqSe8lgyIDt_WcTK9BT5gdTmAoIj0J7R&confirm=t&uuid=6cf6894c-929c-4411-948d-0025fb431f5f
To: /content/RDD2022-India.v5-rdd2022-india.yolov8.zip
100%|██████████| 422M/422M [00:03<00:00, 118MB/s]


'/content/RDD2022-India.v5-rdd2022-india.yolov8.zip'

Data verify and extract

In [ ]:
import os
import zipfile

zip_path = "/content/RDD2022-India.v5-rdd2022-india.yolov8.zip"

print("File exists:", os.path.exists(zip_path))
print("File size (MB):", round(os.path.getsize(zip_path) / (1024 * 1024), 2))

with zipfile.ZipFile(zip_path, "r") as z:
    print("ZIP integrity:", z.testzip())

File exists: True
File size (MB): 402.29
ZIP integrity: None


In [ ]:
import zipfile
import os

zip_path = "/content/RDD2022-India.v5-rdd2022-india.yolov8.zip"
extract_path = "/content/road_damage_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")
print("\nDataset folders:")

for item in os.listdir(extract_path):
    print(item)

Dataset extracted successfully!

Dataset folders:
data.yaml
README.dataset.txt
test
valid
train
README.roboflow.txt


Read Dataset Configuration


In [ ]:
import yaml

yaml_path = "/content/road_damage_dataset/data.yaml"

with open(yaml_path, "r") as file:
    data = yaml.safe_load(file)

print("Dataset configuration:")
print(data)

Dataset configuration:
{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 10, 'names': ['D00', 'D01', 'D0w0', 'D10', 'D11', 'D20', 'D40', 'D43', 'D44', 'D50'], 'roboflow': {'workspace': 'prakhar-kpb1v', 'project': 'rdd2022-india-il8ju', 'version': 5, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/prakhar-kpb1v/rdd2022-india-il8ju/dataset/5'}}


In [ ]:
import os
from collections import Counter

base_path = "/content/road_damage_dataset"

for split in ["train", "valid", "test"]:
    label_path = os.path.join(base_path, split, "labels")

    counts = Counter()

    for file in os.listdir(label_path):
        if file.endswith(".txt"):
            with open(os.path.join(label_path, file)) as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        counts[parts[0]] += 1

    print(f"\n{split.upper()} SET")
    for class_id, count in sorted(counts.items()):
        print(f"Class {class_id}: {count} annotations")


TRAIN SET
Class 0: 1316 annotations
Class 1: 164 annotations
Class 2: 1 annotations
Class 3: 58 annotations
Class 4: 35 annotations
Class 5: 1706 annotations
Class 6: 2680 annotations
Class 7: 55 annotations
Class 8: 909 annotations
Class 9: 24 annotations

VALID SET
Class 0: 172 annotations
Class 1: 10 annotations
Class 3: 5 annotations
Class 4: 8 annotations
Class 5: 215 annotations
Class 6: 340 annotations
Class 7: 2 annotations
Class 8: 113 annotations
Class 9: 2 annotations

TEST SET
Class 0: 67 annotations
Class 1: 5 annotations
Class 3: 5 annotations
Class 4: 2 annotations
Class 5: 100 annotations
Class 6: 167 annotations
Class 8: 40 annotations
Class 9: 2 annotations


Targeted Classes

In [ ]:
import os

base_path = "/content/road_damage_dataset"
target_classes = {"0", "3", "5", "6"}

for split in ["train", "valid", "test"]:
    label_dir = os.path.join(base_path, split, "labels")

    total = 0
    useful = 0

    for filename in os.listdir(label_dir):
        if not filename.endswith(".txt"):
            continue

        total += 1

        with open(os.path.join(label_dir, filename)) as f:
            classes = {
                line.split()[0]
                for line in f if line.strip()
            }

        if classes & target_classes:
            useful += 1

    print(f"\n{split.upper()}")
    print("Total images:", total)
    print("Images containing our 4 classes:", useful)


TRAIN
Total images: 6532
Images containing our 4 classes: 2738

VALID
Total images: 786
Images containing our 4 classes: 338

TEST
Total images: 388
Images containing our 4 classes: 147


In [ ]:
import os
import shutil
import yaml
from pathlib import Path

source = Path("/content/road_damage_dataset")
destination = Path("/content/road_damage_4class")

class_mapping = {
    "0": "0",
    "3": "1",
    "5": "2",
    "6": "3"
}

image_extensions = [".jpg", ".jpeg", ".png", ".bmp"]

for split in ["train", "valid", "test"]:

    source_images = source / split / "images"
    source_labels = source / split / "labels"

    dest_images = destination / split / "images"
    dest_labels = destination / split / "labels"

    dest_images.mkdir(parents=True, exist_ok=True)
    dest_labels.mkdir(parents=True, exist_ok=True)

    kept = 0

    for label_file in source_labels.glob("*.txt"):

        with open(label_file, "r") as f:
            lines = f.readlines()

        new_lines = []

        for line in lines:
            parts = line.strip().split()

            if parts and parts[0] in class_mapping:
                parts[0] = class_mapping[parts[0]]
                new_lines.append(" ".join(parts) + "\n")

        if not new_lines:
            continue

        image_file = None

        for ext in image_extensions:
            candidate = source_images / (label_file.stem + ext)
            if candidate.exists():
                image_file = candidate
                break

        if image_file is None:
            continue

        shutil.copy2(image_file, dest_images / image_file.name)

        with open(dest_labels / label_file.name, "w") as f:
            f.writelines(new_lines)

        kept += 1

    print(f"{split.upper()}: {kept} images copied")

# Create the new YOLO configuration
config = {
    "path": str(destination),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 4,
    "names": ["D00", "D10", "D20", "D40"]
}

with open(destination / "data.yaml", "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("\nFour-class dataset created successfully!")

TRAIN: 2738 images copied
VALID: 338 images copied
TEST: 147 images copied

Four-class dataset created successfully!


Dataset Validation

In [ ]:
import os
from pathlib import Path
from collections import Counter

base = Path("/content/road_damage_4class")

for split in ["train", "valid", "test"]:

    image_dir = base / split / "images"
    label_dir = base / split / "labels"

    images = list(image_dir.glob("*"))
    labels = list(label_dir.glob("*.txt"))

    counts = Counter()
    invalid = 0
    missing = 0

    for label in labels:
        image_exists = any(
            (image_dir / (label.stem + ext)).exists()
            for ext in [".jpg", ".jpeg", ".png", ".bmp"]
        )

        if not image_exists:
            missing += 1

        with open(label) as f:
            for line in f:
                parts = line.strip().split()

                if len(parts) != 5:
                    invalid += 1
                    continue

                class_id = int(parts[0])
                values = list(map(float, parts[1:]))

                if class_id not in [0, 1, 2, 3]:
                    invalid += 1

                if not all(0 <= v <= 1 for v in values):
                    invalid += 1

                counts[class_id] += 1

    print(f"\n{split.upper()}")
    print("Images:", len(images))
    print("Labels:", len(labels))
    print("Missing image pairs:", missing)
    print("Invalid annotations:", invalid)
    print("Class distribution:", dict(sorted(counts.items())))


TRAIN
Images: 2738
Labels: 2738
Missing image pairs: 0
Invalid annotations: 0
Class distribution: {0: 1316, 1: 58, 2: 1706, 3: 2680}

VALID
Images: 338
Labels: 338
Missing image pairs: 0
Invalid annotations: 0
Class distribution: {0: 172, 1: 5, 2: 215, 3: 340}

TEST
Images: 147
Labels: 147
Missing image pairs: 0
Invalid annotations: 0
Class distribution: {0: 67, 1: 5, 2: 100, 3: 167}


Install ultralytics


In [ ]:
!pip install -q ultralytics

YOLOv8 Model Training

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8s model
model = YOLO("yolov8s.pt")

# Train on our four-class road damage dataset
results = model.train(
    data="/content/road_damage_4class/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    seed=42,
    project="/content/road_damage_runs",
    name="rdd_yolov8s_4class",
    patience=10,
    plots=True
)

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/road_damage_4class/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=rdd_yolov8s_4class, nbs=64, nms=None, opset=None, 

Generate predictions for all images

In [ ]:
from pathlib import Path
import cv2

dataset_path = Path("/content/road_damage_4class")
output_path = Path("/content/all_dataset_predictions")

splits = ["train", "valid", "test"]

total_processed = 0

for split in splits:

    print(f"\nProcessing {split.upper()} images...")

    image_folder = dataset_path / split / "images"
    save_folder = output_path / split
    save_folder.mkdir(parents=True, exist_ok=True)

    image_files = sorted([
        p for p in image_folder.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ])

    processed = 0

    # Process images and save each prediction separately
    results = best_model.predict(
        source=str(image_folder),
        conf=0.25,
        stream=True,
        save=False,
        verbose=False
    )

    for result in results:

        image_name = Path(result.path).stem
        annotated_image = result.plot()

        save_file = save_folder / f"{image_name}_prediction.jpg"

        cv2.imwrite(str(save_file), annotated_image)

        processed += 1
        total_processed += 1

        if processed % 100 == 0:
            print(f"{split}: {processed}/{len(image_files)} completed")

    print(f"{split.upper()} completed: {processed} images")

print("\nALL PREDICTIONS COMPLETED!")
print("Total images processed:", total_processed)
print("Saved in:", output_path)


Processing TRAIN images...
train: 100/2738 completed
train: 200/2738 completed
train: 300/2738 completed
train: 400/2738 completed
train: 500/2738 completed
train: 600/2738 completed
train: 700/2738 completed
train: 800/2738 completed
train: 900/2738 completed
train: 1000/2738 completed
train: 1100/2738 completed
train: 1200/2738 completed
train: 1300/2738 completed
train: 1400/2738 completed
train: 1500/2738 completed
train: 1600/2738 completed
train: 1700/2738 completed
train: 1800/2738 completed
train: 1900/2738 completed
train: 2000/2738 completed
train: 2100/2738 completed
train: 2200/2738 completed
train: 2300/2738 completed
train: 2400/2738 completed
train: 2500/2738 completed
train: 2600/2738 completed
train: 2700/2738 completed
TRAIN completed: 2738 images

Processing VALID images...
valid: 100/338 completed
valid: 200/338 completed
valid: 300/338 completed
VALID completed: 338 images

Processing TEST images...
test: 100/147 completed
TEST completed: 147 images

ALL PREDICTIO

In [ ]:
import shutil
from google.colab import files

prediction_zip = shutil.make_archive(
    "/content/all_dataset_predictions",
    "zip",
    "/content/all_dataset_predictions"
)

print("ZIP created:", prediction_zip)

files.download(prediction_zip)

ZIP created: /content/all_dataset_predictions.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Model evaluation on test data


In [ ]:
test_results = best_model.val(
    data="/content/road_damage_4class/data.yaml",
    split="test",
    plots=True
)

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1413.9±522.6 MB/s, size: 56.8 KB)
val: Scanning /content/road_damage_4class/test/labels... 147 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 147/147 1.2Kit/s 0.1s
val: New cache created: /content/road_damage_4class/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 2.7it/s 3.7s
                   all        147        339      0.351      0.439      0.389      0.153
                   D00         48         67       0.35      0.433      0.376      0.154
                   D10          5          5      0.171        0.2      0.162     0.0228
                   D20         86        100      0.566      0.678      0.634      0.291
                   D40         75        167      0.318      0

Evaluation Backup

In [ ]:
import shutil
from google.colab import files

evaluation_folder = "/content/runs/detect/val"

evaluation_zip = shutil.make_archive(
    "/content/road_damage_test_evaluation",
    "zip",
    evaluation_folder
)

print("Evaluation backup created!")

files.download(evaluation_zip)

Evaluation backup created!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download best.pt

In [ ]:
from google.colab import files
import shutil

source = "/content/restored_training/weights/best.pt"
destination = "/content/best.pt"

shutil.copy2(source, destination)

print("Model ready for download!")
files.download(destination)

Model ready for download!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>